# **Scrapping Data**

Scraping data adalah teknik otomatisasi menggunakan bot atau skrip komputer untuk mengekstrak dan mengambil data spesifik dari berbagai sumber seperti halaman website, basis data, atau aplikasi lalu menyimpannya ke dalam format terstruktur seperti spreadsheet atau CSV. 

Cara Kerja Scraping DataProses pengambilan data ini umumnya melalui tiga tahapan utama:
- Request: Program mengirimkan permintaan (command) untuk mengakses halaman atau sumber data yang dituju.
- Parse: Sistem membaca struktur kode (seperti HTML) atau antarmuka untuk mencari dan mengidentifikasi data spesifik yang diinginkan.
- Display/Save: Data yang ditemukan diekstrak dan diubah menjadi format rapi seperti CSV, JSON, atau database agar mudah dianalisis.

## **Persiapan Library**
Memasang pustaka yang diperlukan (trafilatura untuk ekstraksi konten teks bersih, beautifulsoup4 untuk mengambil daftar tautan URL).

In [5]:
!pip install scikit-learn

   ---------------------------------------- 0.0/8.3 MB ? eta -:--:--
   - -------------------------------------- 0.3/8.3 MB ? eta -:--:--
   --- ------------------------------------ 0.8/8.3 MB 2.4 MB/s eta 0:00:04
   ------ --------------------------------- 1.3/8.3 MB 2.2 MB/s eta 0:00:04
   -------- ------------------------------- 1.8/8.3 MB 2.3 MB/s eta 0:00:03
   ----------- ---------------------------- 2.4/8.3 MB 2.3 MB/s eta 0:00:03
   ------------- -------------------------- 2.9/8.3 MB 2.3 MB/s eta 0:00:03
   ---------------- ----------------------- 3.4/8.3 MB 2.4 MB/s eta 0:00:03
   -------------------- ------------------- 4.2/8.3 MB 2.5 MB/s eta 0:00:02
   --------------------- ------------------ 4.5/8.3 MB 2.5 MB/s eta 0:00:02
   ------------------------- -------------- 5.2/8.3 MB 2.5 MB/s eta 0:00:02
   --------------------------- ------------ 5.8/8.3 MB 2.5 MB/s eta 0:00:01
   ------------------------------ --------- 6.3/8.3 MB 2.5 MB/s eta 0:00:01
   -----------------------

In [1]:
!pip install trafilatura beautifulsoup4 pandas requests

## **Langkah 1: Mengumpulkan 200 URL Artikel Detik.com**
Karena Trafilatura fokus mengekstrak isi halaman, kita perlu mengambil daftar URL terlebih dahulu dari halaman indeks berita Detik (misalnya kanal news). Kode ini akan melakukan paging otomatis hingga terkumpul minimal 100 URL unik

In [2]:
import requests
from bs4 import BeautifulSoup
import time
import pandas as pd

def get_detik_urls_by_category(base_url, target_count=100, category_name=""):
    urls = []
    page = 1
    
    print(f"Mengumpulkan URL untuk kategori: {category_name.upper()}...")
    while len(urls) < target_count:
        index_url = f"{base_url}?page={page}"
        headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"}
        
        response = requests.get(index_url, headers=headers)
        if response.status_code != 200:
            print(f"Gagal mengakses halaman {page} untuk {category_name}")
            break
            
        soup = BeautifulSoup(response.text, 'html.parser')
        articles = soup.find_all('article')
        
        for article in articles:
            link = article.find('a')
            if link and 'href' in link.attrs:
                href = link['href']
                if "detik.com" in href and href not in [item['url'] for item in urls]:
                    urls.append({
                        'url': href,
                        'category': category_name
                    })
                    if len(urls) >= target_count:
                        break
                        
        print(f"Kategori {category_name} - Halaman {page}: Terkumpul {len(urls)} URL")
        page += 1
        time.sleep(1) # Jeda agar tidak diblokir server
        
    return urls[:target_count]

# 1. Kumpulkan 100 URL Sport
sport_urls = get_detik_urls_by_category(
    base_url="https://sport.detik.com/indeks", 
    target_count=100, 
    category_name="sport"
)

# 2. Kumpulkan 100 URL Finance
finance_urls = get_detik_urls_by_category(
    base_url="https://finance.detik.com/indeks", 
    target_count=100, 
    category_name="finance"
)

# Gabungkan menjadi satu list total 200 URL
all_target_urls = sport_urls + finance_urls
print(f"\nTotal keseluruhan URL terkumpul: {len(all_target_urls)} URL (100 Sport, 100 Finance).")

Mengumpulkan URL untuk kategori: SPORT...
Kategori sport - Halaman 1: Terkumpul 20 URL
Kategori sport - Halaman 2: Terkumpul 40 URL
Kategori sport - Halaman 3: Terkumpul 60 URL
Kategori sport - Halaman 4: Terkumpul 80 URL
Kategori sport - Halaman 5: Terkumpul 100 URL
Mengumpulkan URL untuk kategori: FINANCE...
Kategori finance - Halaman 1: Terkumpul 20 URL
Kategori finance - Halaman 2: Terkumpul 40 URL
Kategori finance - Halaman 3: Terkumpul 60 URL
Kategori finance - Halaman 4: Terkumpul 80 URL
Kategori finance - Halaman 5: Terkumpul 100 URL

Total keseluruhan URL terkumpul: 200 URL (100 Sport, 100 Finance).


## **Langkah 2: Proses Scrapping Menggunakan Trafilatura**
Setelah daftar URL terkumpul, iterasi setiap URL untuk mengekstrak judul, tanggal, penulis, dan teks isi berita secara bersih menggunakan fungsi trafilatura.

In [3]:
import trafilatura
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time

data_hasil = []
total_data = len(all_target_urls)

print(f"Memulai scraping {total_data} data (Sport & Finance)...")
for idx, item in enumerate(all_target_urls):
    url = item['url']
    category = item['category']
    
    try:
        # 1. Ambil HTML mentah menggunakan requests untuk parsing judul & tanggal
        headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"}
        resp = requests.get(url, headers=headers)
        
        title = None
        date = None
        if resp.status_code == 200:
            soup = BeautifulSoup(resp.text, 'html.parser')
            # Ambil judul dari tag <h1> atau <title>
            title_tag = soup.find('h1') or soup.find('title')
            if title_tag:
                title = title_tag.text.strip()
                
            # Ambil tanggal dari tag div class detail__date
            date_tag = soup.find('div', class_='detail__date')
            if date_tag:
                date = date_tag.text.strip()

        # 2. Ambil teks bersih menggunakan Trafilatura
        downloaded = trafilatura.fetch_url(url)
        text = ""
        if downloaded:
            extracted = trafilatura.extract(downloaded, include_comments=False, include_tables=False)
            if extracted:
                text = extracted
                
        # Simpan ke list of dictionary
        data_hasil.append({
            'url': url,
            'category': category, # Menyimpan label sport atau finance
            'title': title,
            'date': date,
            'text': text
        })
        
        print(f"Berhasil scrape [{idx+1}/{total_data}] - {category.upper()}")
    except Exception as e:
        print(f"Gagal pada URL {url}: {e}")
        
    time.sleep(0.5)

# 3. Buat DataFrame
df = pd.DataFrame(data_hasil)

# 4. Tambahkan kolom panjang karakter dengan perlindungan fillna('') agar terhindar dari TypeError
df['title_length'] = df['title'].fillna('').astype(str).apply(len)
df['text_length'] = df['text'].fillna('').astype(str).apply(len)



Memulai scraping 200 data (Sport & Finance)...
Berhasil scrape [1/200] - SPORT
Berhasil scrape [2/200] - SPORT
Berhasil scrape [3/200] - SPORT
Berhasil scrape [4/200] - SPORT
Berhasil scrape [5/200] - SPORT
Berhasil scrape [6/200] - SPORT
Berhasil scrape [7/200] - SPORT
Berhasil scrape [8/200] - SPORT
Berhasil scrape [9/200] - SPORT
Berhasil scrape [10/200] - SPORT
Berhasil scrape [11/200] - SPORT
Berhasil scrape [12/200] - SPORT
Berhasil scrape [13/200] - SPORT
Berhasil scrape [14/200] - SPORT
Berhasil scrape [15/200] - SPORT
Berhasil scrape [16/200] - SPORT
Berhasil scrape [17/200] - SPORT
Berhasil scrape [18/200] - SPORT
Berhasil scrape [19/200] - SPORT
Berhasil scrape [20/200] - SPORT
Berhasil scrape [21/200] - SPORT
Berhasil scrape [22/200] - SPORT
Berhasil scrape [23/200] - SPORT
Berhasil scrape [24/200] - SPORT
Berhasil scrape [25/200] - SPORT
Berhasil scrape [26/200] - SPORT
Berhasil scrape [27/200] - SPORT
Berhasil scrape [28/200] - SPORT
Berhasil scrape [29/200] - SPORT
Berha

## **Langkah 3: Menampilkan Hasil**
Menampilkan beberapa sampel data teratas untuk memastikan hasil ekstraksi bersih.

In [4]:
# Tampilkan sampel data teratas (Sport) dan terbawah (Finance)
print("\n--- SAMPEL DATA SPORT ---")
display(df.head(3))

print("\n--- SAMPEL DATA FINANCE ---")
display(df.tail(3))



--- SAMPEL DATA SPORT ---


,url,category,title,date,text,title_length,text_length
0,https://sport.detik.com/raket/d-8656258/mens-w...,sport,Men's World Tennis Championship: Rifqi dan Raf...,"Kamis, 10 Sep 2026 04:42 WIB","Dua petenis Indonesia, Muhammad Rifqi Fitriadi...",73,1489
1,https://sport.detik.com/raket/d-8656218/kevin-...,sport,Kevin Sanjaya Antusias Lihat Semangat Talenta ...,"Kamis, 10 Sep 2026 01:15 WIB",Audisi Umum PB Djarum 2026 sedang berlangsung ...,67,1829
2,https://sport.detik.com/sport-lain/d-8656170/a...,sport,Atlet Muda Modern Pentathlon Punya Kesempatan ...,"Rabu, 09 Sep 2026 23:50 WIB",Modern Pentathlon akan dipertandingkan dalam m...,63,2861



--- SAMPEL DATA FINANCE ---


,url,category,title,date,text,title_length,text_length
197,https://finance.detik.com/energi/d-8654238/per...,finance,"Pertamina Waspada Solar Subsidi 'Bocor', Selis...","Selasa, 08 Sep 2026 17:53 WIB",PT Pertamina Patra Niaga mewaspadai adanya pot...,71,1682
198,https://finance.detik.com/berita-ekonomi-bisni...,finance,TikTok & Tokopedia Buka Suara soal Kabar Bloki...,"Selasa, 08 Sep 2026 17:43 WIB",Tokopedia dan TikTok Shop buka suara terkait l...,66,3217
199,https://finance.detik.com/berita-ekonomi-bisni...,finance,"Purbaya Ngaku Tak Pusingkan Anggaran BGN, Tahu...","Selasa, 08 Sep 2026 17:32 WIB",Menteri Keuangan (Menkeu) Purbaya Yudhi Sadewa...,71,1889


## **Data Understanding**

In [5]:
import pandas as pd
import numpy as np

# 1. Menampilkan 5 baris pertama dan 5 baris terakhir data
print("--- 5 BARIS PERTAMA DATA ---")
display(df.head())
print("\n--- 5 BARIS TERAKHIR DATA ---")
display(df.tail())

# 2. Informasi Umum Dataset (Jumlah baris, kolom, tipe data, & memory usage)
print("\n--- INFORMASI UMUM DATASET ---")
df.info()

# 3. Pengecekan Missing Value (Nilai yang Kosong)
print("\n--- JUMLAH MISSING VALUE PER KOLOM ---")
missing_data = df.isnull().sum()
print(missing_data)

# 4. Distribusi Jumlah Data Berdasarkan Kategori
print("\n--- DISTRIBUSI KATEGORI (SPORT & FINANCE) ---")
category_counts = df['category'].value_counts()
print(category_counts)

# 5. Statistik Deskriptif untuk Panjang Karakter Teks & Judul
print("\n--- STATISTIK DESKRIPTIF PANJANG TEKS & JUDUL ---")
display(df[['title_length', 'text_length']].describe())

# 6. Pengecekan Duplikasi Data (Berdasarkan URL atau Judul)
print("\n--- JUMLAH DATA DUPLIKAT ---")
print(f"Duplikat berdasarkan URL: {df['url'].duplicated().sum()}")
print(f"Duplikat berdasarkan Judul: {df['title'].duplicated().sum()}")

--- 5 BARIS PERTAMA DATA ---


,url,category,title,date,text,title_length,text_length
0,https://sport.detik.com/raket/d-8656258/mens-w...,sport,Men's World Tennis Championship: Rifqi dan Raf...,"Kamis, 10 Sep 2026 04:42 WIB","Dua petenis Indonesia, Muhammad Rifqi Fitriadi...",73,1489
1,https://sport.detik.com/raket/d-8656218/kevin-...,sport,Kevin Sanjaya Antusias Lihat Semangat Talenta ...,"Kamis, 10 Sep 2026 01:15 WIB",Audisi Umum PB Djarum 2026 sedang berlangsung ...,67,1829
2,https://sport.detik.com/sport-lain/d-8656170/a...,sport,Atlet Muda Modern Pentathlon Punya Kesempatan ...,"Rabu, 09 Sep 2026 23:50 WIB",Modern Pentathlon akan dipertandingkan dalam m...,63,2861
3,https://sport.detik.com/sport-lain/d-8656176/r...,sport,Romy Tahrizi Juara Nasional Shifter 2026 Bersa...,"Rabu, 09 Sep 2026 23:15 WIB",Kejuaraan Nasional Gokart 2026 yang berlangsun...,60,4791
4,https://sport.detik.com/sport-lain/d-8656181/a...,sport,Asian Games: Ketum PBTI Yakin Taekwondo Bisa B...,"Rabu, 09 Sep 2026 22:22 WIB",Asian Games 2026 tak lama lagi akan dimulai. K...,74,1624



--- 5 BARIS TERAKHIR DATA ---


,url,category,title,date,text,title_length,text_length
195,https://finance.detik.com/berita-ekonomi-bisni...,finance,TelkomProperty Ungkap Pemanfaatan Graha Merah ...,"Selasa, 08 Sep 2026 18:50 WIB","PT Graha Sarana Duta (TelkomProperty), sebagai...",76,1948
196,https://finance.detik.com/berita-ekonomi-bisni...,finance,"Harga Masker Mahal Imbas Abu Anak Krakatau, Pe...","Selasa, 08 Sep 2026 17:58 WIB",Ketua Badan Perlindungan Konsumen Nasional (BP...,71,2205
197,https://finance.detik.com/energi/d-8654238/per...,finance,"Pertamina Waspada Solar Subsidi 'Bocor', Selis...","Selasa, 08 Sep 2026 17:53 WIB",PT Pertamina Patra Niaga mewaspadai adanya pot...,71,1682
198,https://finance.detik.com/berita-ekonomi-bisni...,finance,TikTok & Tokopedia Buka Suara soal Kabar Bloki...,"Selasa, 08 Sep 2026 17:43 WIB",Tokopedia dan TikTok Shop buka suara terkait l...,66,3217
199,https://finance.detik.com/berita-ekonomi-bisni...,finance,"Purbaya Ngaku Tak Pusingkan Anggaran BGN, Tahu...","Selasa, 08 Sep 2026 17:32 WIB",Menteri Keuangan (Menkeu) Purbaya Yudhi Sadewa...,71,1889



--- INFORMASI UMUM DATASET ---
<class 'pandas.DataFrame'>
RangeIndex: 200 entries, 0 to 199
Data columns (total 7 columns):
 #   Column        Non-Null Count  Dtype
---  ------        --------------  -----
 0   url           200 non-null    str  
 1   category      200 non-null    str  
 2   title         200 non-null    str  
 3   date          197 non-null    str  
 4   text          200 non-null    str  
 5   title_length  200 non-null    int64
 6   text_length   200 non-null    int64
dtypes: int64(2), str(5)
memory usage: 11.1 KB

--- JUMLAH MISSING VALUE PER KOLOM ---
url             0
category        0
title           0
date            3
text            0
title_length    0
text_length     0
dtype: int64

--- DISTRIBUSI KATEGORI (SPORT & FINANCE) ---
category
sport      100
finance    100
Name: count, dtype: int64

--- STATISTIK DESKRIPTIF PANJANG TEKS & JUDUL ---


,title_length,text_length
count,200.000000,200.000000
mean,59.780000,2344.615000
std,11.747409,1083.145687
min,20.000000,334.000000
25%,52.000000,1677.000000
50%,61.500000,2156.500000
75%,69.000000,2691.250000
max,85.000000,8169.000000



--- JUMLAH DATA DUPLIKAT ---
Duplikat berdasarkan URL: 0
Duplikat berdasarkan Judul: 0


In [6]:
# convert csv
# Menampilkan informasi ringkas setelah digabung
print(f"Total baris dataset gabungan: {len(df)}")
print(f"Distribusi kategori:\n{df['category'].value_counts()}")

# Menyimpan dataset gabungan ke file CSV
df.to_csv('dataset/hasil_scrapping_detik_200.csv', index=False, encoding='utf-8')
print("\nFile berhasil digabung dan disimpan sebagai 'dataset/hasil_scrapping_detik_200.csv'")

# Menampilkan 2 sampel teratas dari dataframe gabungan
display(df.head(2))

Total baris dataset gabungan: 200
Distribusi kategori:
category
sport      100
finance    100
Name: count, dtype: int64

File berhasil digabung dan disimpan sebagai 'dataset/hasil_scrapping_detik_200.csv'


,url,category,title,date,text,title_length,text_length
0,https://sport.detik.com/raket/d-8656258/mens-w...,sport,Men's World Tennis Championship: Rifqi dan Raf...,"Kamis, 10 Sep 2026 04:42 WIB","Dua petenis Indonesia, Muhammad Rifqi Fitriadi...",73,1489
1,https://sport.detik.com/raket/d-8656218/kevin-...,sport,Kevin Sanjaya Antusias Lihat Semangat Talenta ...,"Kamis, 10 Sep 2026 01:15 WIB",Audisi Umum PB Djarum 2026 sedang berlangsung ...,67,1829


## **Preprocessing**

In [1]:
import pandas as pd

# Memuat dataset yang sudah disimpan sebelumnya
# Sesuaikan path jika lokasi folder berbeda (misal: 'hasil_scrapping_detik_200.csv' atau 'dataset/hasil_scrapping_detik_200.csv')
df = pd.read_csv('dataset/hasil_scrapping_detik_200.csv')

print(f"Data berhasil dimuat! Jumlah total baris: {len(df)}")
display(df.head())

Data berhasil dimuat! Jumlah total baris: 200


,url,category,title,date,text,title_length,text_length
0,https://sport.detik.com/raket/d-8656258/mens-w...,sport,Men's World Tennis Championship: Rifqi dan Raf...,"Kamis, 10 Sep 2026 04:42 WIB","Dua petenis Indonesia, Muhammad Rifqi Fitriadi...",73,1489
1,https://sport.detik.com/raket/d-8656218/kevin-...,sport,Kevin Sanjaya Antusias Lihat Semangat Talenta ...,"Kamis, 10 Sep 2026 01:15 WIB",Audisi Umum PB Djarum 2026 sedang berlangsung ...,67,1829
2,https://sport.detik.com/sport-lain/d-8656170/a...,sport,Atlet Muda Modern Pentathlon Punya Kesempatan ...,"Rabu, 09 Sep 2026 23:50 WIB",Modern Pentathlon akan dipertandingkan dalam m...,63,2861
3,https://sport.detik.com/sport-lain/d-8656176/r...,sport,Romy Tahrizi Juara Nasional Shifter 2026 Bersa...,"Rabu, 09 Sep 2026 23:15 WIB",Kejuaraan Nasional Gokart 2026 yang berlangsun...,60,4791
4,https://sport.detik.com/sport-lain/d-8656181/a...,sport,Asian Games: Ketum PBTI Yakin Taekwondo Bisa B...,"Rabu, 09 Sep 2026 22:22 WIB",Asian Games 2026 tak lama lagi akan dimulai. K...,74,1624


### **Cleaning Data**

In [9]:
from sklearn.feature_extraction.text import CountVectorizer
import re
import string

# Daftar stopword sederhana Bahasa Indonesia untuk menyaring kata tugas
stopwords_indonesia = set([
    'dan', 'yang', 'di', 'ke', 'dari', 'ini', 'itu', 'untuk',
    'pada', 'dengan', 'adakah', 'sebagai', 'oleh', 'karena',
    'bahwa', 'atau', 'dalam', 'mereka', 'nya', 'tersebut',
    'juga', 'akan', 'terhadap', 'secara', 'dapat', 'bagi',
    'sudah'
])

def advanced_preprocessing(text):
    if pd.isna(text):
        return ""

    # 1. Case Folding (Ubah ke huruf kecil semua)
    text = text.lower()
    # 2. Hapus tautan/URL
    text = re.sub(r'https?://\S+|www\.\S+', '', text)
    # 3. Hapus angka
    text = re.sub(r'\d+', '', text)
    # 4. Hapus tanda baca (titik, koma, tanda seru, dll)
    text = text.translate(str.maketrans('', '', string.punctuation))
    
    # Hapus stopword dan kata yang terlalu pendek (1-2 huruf)
    words = text.split()
    filtered_words = [w for w in words if w not in stopwords_indonesia and len(w) > 2]
    
    return " ".join(filtered_words)

# Terapkan ulang preprocessing yang lebih bersih
df['clean_text'] = df['text'].apply(advanced_preprocessing)
print(" hasil teks setelah preprocessing:")
display(df[['text', 'clean_text']].head(4))

 hasil teks setelah preprocessing:


,text,clean_text
0,"Dua petenis Indonesia, Muhammad Rifqi Fitriadi...",dua petenis indonesia muhammad rifqi fitriadi ...
1,Audisi Umum PB Djarum 2026 sedang berlangsung ...,audisi umum djarum sedang berlangsung kudus ke...
2,Modern Pentathlon akan dipertandingkan dalam m...,modern pentathlon dipertandingkan momen hari u...
3,Kejuaraan Nasional Gokart 2026 yang berlangsun...,kejuaraan nasional gokart berlangsung enam put...


### **Ekstrak kata unik**

In [12]:
from sklearn.feature_extraction.text import CountVectorizer
import pandas as pd

# 1. Inisialisasi CountVectorizer untuk mengekstrak seluruh kata unik
# min_df=2 artinya kata harus muncul minimal di 2 dokumen agar dicatat sebagai fitur
vectorizer = CountVectorizer(min_df=2)

# 2. Proses ekstraksi kata unik dari kolom 'clean_text'
X_count = vectorizer.fit_transform(df['clean_text'])
# 3. Ubah hasil sparse matrix menjadi Pandas DataFrame (kolom = kata unik, baris = dokumen)
df_bow = pd.DataFrame(X_count.toarray(), columns=vectorizer.get_feature_names_out())
# 4. Tambahkan kolom label sesuai permintaan (sport: 1, finance: 2)
df_bow['label'] = df['category'].map({'sport': 1, 'finance': 2})

print(f"Jumlah dokumen: {df_bow.shape[0]} baris")
print(f"Jumlah kata unik setelah difilter: {df_bow.shape[1] - 1} kata unik")

# Tampilkan beberapa kolom pertama dan kolom label terakhir dari matriks
display(df_bow.iloc[:, :10].assign(label=df_bow['label']).head())

Jumlah dokumen: 200 baris
Jumlah kata unik setelah difilter: 3565 kata unik


,aau,abdul,absen,abu,acara,accurate,acdacd,aceh,acehsumut,achmad,label
0,0,0,0,0,0,0,0,0,0,0,1
1,0,0,0,0,0,0,0,0,0,0,1
2,1,0,0,0,0,0,0,0,0,0,1
3,0,0,0,1,0,0,0,0,0,0,1
4,0,0,0,0,1,0,0,0,0,0,1


### **Penerapan TF-IDF**

In [16]:
from sklearn.feature_extraction.text import TfidfVectorizer
import pandas as pd

# 1. Inisialisasi TfidfVectorizer
# Kita menggunakan parameter min_df=2 yang sama seperti sebelumnya agar kata yang terlalu langka tersaring
tfidf_vectorizer = TfidfVectorizer(min_df=2)

# 2. Fit and transform teks yang sudah dibersihkan (clean_text)
X_tfidf = tfidf_vectorizer.fit_transform(df['clean_text'])

# 3. Ubah hasil matrix sparse TF-IDF menjadi Pandas DataFrame
df_tfidf = pd.DataFrame(X_tfidf.toarray(), columns=tfidf_vectorizer.get_feature_names_out())

# 4. Tambahkan kembali kolom label (sport: 1, finance: 2) di ujung DataFrame
df_tfidf['label'] = df['category'].map({'sport': 1, 'finance': 2})

print("Representasi TF-IDF berhasil dibuat!")
print(f"Ukuran Dataset TF-IDF: {df_tfidf.shape[0]} baris dokumen dan {df_tfidf.shape[1] - 1} kolom kata unik (+ 1 kolom label)")

# Tampilkan 5 baris pertama dan terakhir dari beberapa kolom kata pertama dan kolom label
display(df_tfidf.iloc[:, :8].assign(label=df_tfidf['label']).head())
display(df_tfidf.iloc[:, :8].assign(label=df_tfidf['label']).tail())

Representasi TF-IDF berhasil dibuat!
Ukuran Dataset TF-IDF: 200 baris dokumen dan 3565 kolom kata unik (+ 1 kolom label)


,aau,abdul,absen,abu,acara,accurate,acdacd,aceh,label
0,0.000000,0.0,0.0,0.00000,0.000000,0.0,0.0,0.0,1
1,0.000000,0.0,0.0,0.00000,0.000000,0.0,0.0,0.0,1
2,0.041044,0.0,0.0,0.00000,0.000000,0.0,0.0,0.0,1
3,0.000000,0.0,0.0,0.03297,0.000000,0.0,0.0,0.0,1
4,0.000000,0.0,0.0,0.00000,0.067224,0.0,0.0,0.0,1


,aau,abdul,absen,abu,acara,accurate,acdacd,aceh,label
195,0.0,0.0,0.0,0.000000,0.0,0.0,0.000000,0.0,2
196,0.0,0.0,0.0,0.106939,0.0,0.0,0.043325,0.0,2
197,0.0,0.0,0.0,0.000000,0.0,0.0,0.000000,0.0,2
198,0.0,0.0,0.0,0.000000,0.0,0.0,0.036454,0.0,2
199,0.0,0.0,0.0,0.000000,0.0,0.0,0.051251,0.0,2
